# School → GP Mapping & GP-Level Education Aggregation

Standalone notebook. Run this **separately** from `02_gp_level_join.ipynb`, then feed its
final output (`devadurga_gp_education_data.csv`) back into the master join notebook.

**Pipeline:**

```
446 schools -> exact/verified School->GP mapping -> 33 GP aggregation -> Education_Data -> merge with Master
```

**Golden rule: never auto-guess a GP from village name alone.** A wrong GP assignment
directly corrupts the Need Score. This notebook only auto-maps *obvious, safe* matches
and produces a review file for everything else — you fill in the rest by hand using
UDISE+ (`Know Your School`) lookups, using the **LGD Panchayat** field as the source of
truth (village name != GP name; e.g. village `Madarkal` sits in GP `Shavantagera`).

**Steps in this notebook:**
1. Load the 446 cleaned school records
2. Load the 33 target GPs (from your master table CSV)
3. Normalize names for matching
4. Auto-map exact `village == GP name` matches
5. Find GP-name candidates inside the `Address` field
6. Auto-accept address matches only when exactly one candidate exists
7. Export a review file — unresolved schools flagged for manual lookup
8. **(Manual step, outside this notebook)** fill in the blanks using UDISE+
9. Reload the completed mapping file
10. Validate it (no unmapped schools, no invalid GPs, no duplicate UDISE codes, LGD codes consistent)
11. Aggregate 446 schools -> 33 GP-level education indicators
12. Add derived indicators (students/teacher, coverage %, etc.)
13. Confirm all 33 target GPs are present
14. Save `devadurga_gp_education_data.csv` — this is the file `02_gp_level_join.ipynb` merges in

---

## 0. Setup — mount Drive and set paths

Edit `BASE_DIR` if your project lives elsewhere.

In [ ]:
# Mount Google Drive (Colab only — skip/comment out if running locally)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os
import re

# ---- Project folders ----
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Data_Organized"
CLEAN_DIR = os.path.join(BASE_DIR, "Cleaned")

# ---- Input files ----
SCHOOL_FILE = os.path.join(CLEAN_DIR, "Devadurga_446_Schools_Cleaned.xlsx")

# The 33-GP master table produced by 02_gp_level_join.ipynb.
# If you already have `merged` in memory (ran this in the same session as
# notebook 02), that will be used instead automatically — see Cell 2 below.
MASTER_GP_FILE = os.path.join(CLEAN_DIR, "gp_level_master_joined.csv")

# ---- Output files this notebook produces ----
REVIEW_FILE = os.path.join(CLEAN_DIR, "Devadurga_School_GP_Mapping_Review.xlsx")
COMPLETED_MAPPING_FILE = os.path.join(CLEAN_DIR, "Devadurga_School_GP_Mapping_Completed.xlsx")
EDUCATION_OUTPUT = os.path.join(CLEAN_DIR, "devadurga_gp_education_data.csv")

print("CLEAN_DIR:", CLEAN_DIR)

CLEAN_DIR: /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned


## 1. Load the 446 cleaned school records

In [ ]:
schools = pd.read_excel(SCHOOL_FILE, sheet_name="Cleaned_446_Schools")

print("School data shape:", schools.shape)
print("Number of unique UDISE codes:", schools["UDISE Code"].nunique())

display(schools.head())

# Expected:
# School data shape: (446, 53)
# Number of unique UDISE codes: 446

School data shape: (446, 53)
Number of unique UDISE codes: 446


,UDISE Code,School Name,Year Established,Address,Village / Locality,Pincode,Block,District,State,School Category,...,VI,VII,VIII,IX,X,XI,XII,Enrollment Total,Source URL,Fetch Status
0,29060101215,Adarsa Vidyalaya(Rmsa) Arakera,2010,584111) Arkera Devadurga Raichur Karnataka.,Arkera,584111.0,Devadurga,Raichur,Karnataka,7-Upper Pr. and Secondary,...,74,66,63,55,44,0,0,302,https://stackschools.com/schools/29060101215/a...,OK
1,29060121903,Ashrama Schoolsociel Welfer De,1994,584111) Devadurga City Devadurga Raichur Karna...,Devadurga City,584111.0,Devadurga,Raichur,Karnataka,1-Primary,...,0,0,0,0,0,0,0,131,https://stackschools.com/schools/29060121903/a...,OK
2,29060122403,A Venkatesh Nayak (Puc) Deodurga,2005,584111) Devadurga City Devadurga Raichur Karna...,Devadurga City,584111.0,Devadurga,Raichur,Karnataka,11-Higher Secondary only/Jr. College,...,0,0,0,0,0,123,86,209,https://stackschools.com/schools/29060122403/a...,OK
3,29060122402,Basava Hs Deodurga,2001,584111) Devadurga City Devadurga Raichur Karna...,Devadurga City,584111.0,Devadurga,Raichur,Karnataka,8-Secondary Only,...,0,0,39,48,21,0,0,108,https://stackschools.com/schools/29060122402/b...,OK
4,29060121718,Basava Indp Pu College Devadurga,2002,584111) Alkod Devadurga Raichur Karnataka.,Alkod,584111.0,Devadurga,Raichur,Karnataka,11-Higher Secondary only/Jr. College,...,0,0,0,0,0,273,174,447,https://stackschools.com/schools/29060121718/b...,OK


## 2. Load the 33 target Gram Panchayats

Works two ways:
- If `merged` already exists in this Python session (you ran `02_gp_level_join.ipynb`
  in the same runtime first), it's used directly.
- Otherwise it's loaded fresh from the saved `gp_level_master_joined.csv`.

In [ ]:
if "merged" in globals():
    gp_source = merged
    print("Using in-memory `merged` table from 02_gp_level_join.ipynb")
else:
    gp_source = pd.read_csv(MASTER_GP_FILE)
    print("Loaded master GP table from disk:", MASTER_GP_FILE)

gp_master = (
    gp_source[["gp_name", "gp_lgd_code"]]
    .drop_duplicates()
    .copy()
)

print("Number of target GPs:", len(gp_master))
display(gp_master.sort_values("gp_name"))

assert len(gp_master) == 33, f"Expected 33 target GPs, got {len(gp_master)} — check the master table."

Loaded master GP table from disk: /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/gp_level_master_joined.csv
Number of target GPs: 33


,gp_name,gp_lgd_code
0,Alkod,219671
1,Amarapur,273065
2,Arkera,219672
3,B.Ganekal,219673
4,Bhumanagunda,273066
5,Chinchodi,219674
6,Dondambli,219675
7,Gabbur,219676
8,Galaga,219677
9,Ganadhal (Deodurga),219678


## 3. Normalize names for matching

Makes `B. Ganekal` and `B.Ganekal` (etc.) comparable. This never touches the original
data columns — only the new `*_norm` helper columns.

In [ ]:
def normalize_text(x):
    """Normalize names for matching. Does NOT change the original data."""
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = x.replace(".", " ").replace("-", " ").replace("/", " ")
    x = re.sub(r"[^A-Z0-9 ]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


gp_master["gp_name_norm"] = gp_master["gp_name"].apply(normalize_text)

schools["village_norm"] = schools["Village / Locality"].apply(normalize_text)
schools["address_norm"] = schools["Address"].apply(normalize_text)
schools["school_name_norm"] = schools["School Name"].apply(normalize_text)

print("Normalization completed.")

Normalization completed.


## 4. Automatic exact mapping (village == GP name)

Deliberately conservative: only maps a school when `Village / Locality` matches a
target GP name exactly.

In [ ]:
gp_lookup = dict(zip(gp_master["gp_name_norm"], gp_master["gp_name"]))
gp_code_lookup = dict(zip(gp_master["gp_name_norm"], gp_master["gp_lgd_code"]))

schools["GP Name"] = schools["village_norm"].map(gp_lookup)
schools["GP LGD Code"] = schools["village_norm"].map(gp_code_lookup)

schools["Mapping Method"] = np.where(
    schools["GP Name"].notna(),
    "Exact village = GP",
    "UNRESOLVED"
)

print("Automatically mapped:", schools["GP Name"].notna().sum())
print("Unresolved:", schools["GP Name"].isna().sum())

display(
    schools[[
        "UDISE Code", "School Name", "Village / Locality",
        "GP Name", "GP LGD Code", "Mapping Method"
    ]].head(20)
)

Automatically mapped: 179
Unresolved: 267


,UDISE Code,School Name,Village / Locality,GP Name,GP LGD Code,Mapping Method
0,29060101215,Adarsa Vidyalaya(Rmsa) Arakera,Arkera,Arkera,219672.0,Exact village = GP
1,29060121903,Ashrama Schoolsociel Welfer De,Devadurga City,NaN,NaN,UNRESOLVED
2,29060122403,A Venkatesh Nayak (Puc) Deodurga,Devadurga City,NaN,NaN,UNRESOLVED
3,29060122402,Basava Hs Deodurga,Devadurga City,NaN,NaN,UNRESOLVED
4,29060121718,Basava Indp Pu College Devadurga,Alkod,Alkod,219671.0,Exact village = GP
5,29060121713,Basava Lower Primary School Deodurg,Devadurga City,NaN,NaN,UNRESOLVED
6,29060120802,Basaveshwara High School,Devadurga South,NaN,NaN,UNRESOLVED
7,29060121202,Basaveswar Higher Primary School Devadurga,Devadurga South,NaN,NaN,UNRESOLVED
8,29060104012,Brillant Lps Gabbur,Gabbur,Gabbur,219676.0,Exact village = GP
9,29060121906,Chalukya Vidya Mandira Lps Devadurga,Devadurga City,NaN,NaN,UNRESOLVED


## 5. Find GP-name candidates inside the `Address` field

Some schools carry the GP name in their address even though `Village / Locality`
differs. These are flagged as **candidates only** — nothing is auto-applied yet if
there's ambiguity.

In [ ]:
def find_gp_candidates(text):
    """Find target GP names appearing inside a text field. Returns all matches."""
    if not text:
        return []
    return [gp_name for gp_norm, gp_name in gp_lookup.items() if gp_norm and gp_norm in text]


schools["Address GP Candidates"] = schools["address_norm"].apply(find_gp_candidates)
schools["Address GP Candidate Count"] = schools["Address GP Candidates"].apply(len)

print("Schools with exactly one address GP candidate:",
      (schools["Address GP Candidate Count"] == 1).sum())
print("Schools with multiple address GP candidates:",
      (schools["Address GP Candidate Count"] > 1).sum())

display(
    schools[schools["Address GP Candidate Count"] > 0][[
        "UDISE Code", "School Name", "Village / Locality",
        "Address", "GP Name", "Address GP Candidates"
    ]].head(30)
)

Schools with exactly one address GP candidate: 197
Schools with multiple address GP candidates: 0


,UDISE Code,School Name,Village / Locality,Address,GP Name,Address GP Candidates
0,29060101215,Adarsa Vidyalaya(Rmsa) Arakera,Arkera,584111) Arkera Devadurga Raichur Karnataka.,Arkera,[Arkera]
4,29060121718,Basava Indp Pu College Devadurga,Alkod,584111) Alkod Devadurga Raichur Karnataka.,Alkod,[Alkod]
8,29060104012,Brillant Lps Gabbur,Gabbur,584113) Gabbur Devadurga Raichur Karnataka.,Gabbur,[Gabbur]
11,29060104304,Channabasaveswara Lower Primary School Gmath,Mustur,584126) Mustur Devadurga Raichur Karnataka.,Mustur,[Mustur]
12,29060107825,Darwin Hps Jalahalli,Jalahalli,584116) Jalahalli Devadurga Raichur Karnataka.,Jalahalli,[Jalahalli]
13,29060122105,Don Bosco Pre University College,Alkod,584111) Alkod Devadurga Raichur Karnataka.,Alkod,[Alkod]
19,29060107803,Ghps Horahatti,Jalahalli,584116) Jalahalli Devadurga Raichur Karnataka.,Jalahalli,[Jalahalli]
20,29060112701,Ghps Mundaragi,Mundaragi,584116) Mundaragi Devadurga Raichur Karnataka.,Mundaragi,[Mundaragi]
21,29060112801,Ghps Mustur,Mustur,584126) Mustur Devadurga Raichur Karnataka.,Mustur,[Mustur]
25,29060107812,Ghps Talawar Oni Jalahalli,Jalahalli West,584116) Jalahalli West Devadurga Raichur Karna...,NaN,[Jalahalli]


## 6. Auto-accept only *safe* address matches

If a school is still unmapped **and** has exactly one GP candidate in its address,
that candidate is accepted. Anything with 0 or 2+ candidates stays unresolved for
manual review.

In [ ]:
address_match = (
    schools["GP Name"].isna()
    & (schools["Address GP Candidate Count"] == 1)
)

schools.loc[address_match, "GP Name"] = schools.loc[address_match, "Address GP Candidates"].str[0]

schools["GP LGD Code"] = schools["GP Name"].map(
    dict(zip(gp_master["gp_name"], gp_master["gp_lgd_code"]))
)

schools.loc[address_match, "Mapping Method"] = "Exact GP found in address"

print("Mapped after address check:", schools["GP Name"].notna().sum())
print("Still unresolved:", schools["GP Name"].isna().sum())

Mapped after address check: 197
Still unresolved: 249


In [ ]:
v2g = pd.read_csv(os.path.join(CLEAN_DIR, "raichur_village_to_gp_mapping.csv"))
print(v2g.shape)
print(v2g.columns.tolist())
display(v2g.head(10))

(867, 6)
['sub_taluk', 'village_code', 'village_name', 'coverage_type', 'gp_lgd_code', 'gp_name']


,sub_taluk,village_code,village_name,coverage_type,gp_lgd_code,gp_name
0,Arakera,600702,Adkalgud,FULL,273066,Bhumanagunda
1,Arakera,600715,Agrahar,FULL,273067,Malledevaragudda
2,Arakera,600704,Akalkumpi,FULL,219671,Alkod
3,Arakera,600660,Aldharti,FULL,219695,Mustur
4,Arakera,600706,Alkod,FULL,219671,Alkod
5,Arakera,600663,Andankhed,FULL,219698,Ramadurga
6,Arakera,600703,Anwar,FULL,219671,Alkod
7,Arakera,600697,Arkera,FULL,219672,Arkera
8,Arakera,600709,Bandegud,FULL,273067,Malledevaragudda
9,Arakera,600552,Bassapur,FULL,219682,Hosur Siddapur


In [ ]:
# ============================================================
# CELL 6B — RESOLVE REMAINING SCHOOLS USING AUTHORITATIVE
#           VILLAGE -> GP MAPPING (raichur_village_to_gp_mapping.csv)
# ============================================================

print("coverage_type values:", v2g["coverage_type"].value_counts(dropna=False))

VCOL = "village_name"
GCOL = "gp_name"
GCODECOL = "gp_lgd_code"

v2g["_village_norm"] = v2g[VCOL].apply(normalize_text)
v2g = v2g[v2g[GCOL].isin(gp_master["gp_name"])].copy()

dupe_villages = v2g.groupby("_village_norm")[GCOL].nunique()
ambiguous_villages = dupe_villages[dupe_villages > 1].index.tolist()
if ambiguous_villages:
    print(f"WARNING: {len(ambiguous_villages)} villages map to >1 GP -- left unresolved.")
    print(ambiguous_villages)

v2g_clean = v2g[~v2g["_village_norm"].isin(ambiguous_villages)]
village_to_gp = dict(zip(v2g_clean["_village_norm"], v2g_clean[GCOL]))
village_to_gpcode = dict(zip(v2g_clean["_village_norm"], v2g_clean[GCODECOL]))

still_unresolved = schools["GP Name"].isna()
schools.loc[still_unresolved, "GP Name"] = schools.loc[still_unresolved, "village_norm"].map(village_to_gp)

newly_filled = still_unresolved & schools["GP Name"].notna()
schools.loc[newly_filled, "GP LGD Code"] = schools.loc[newly_filled, "village_norm"].map(village_to_gpcode)
schools.loc[newly_filled, "Mapping Method"] = "Village->GP via raichur_village_to_gp_mapping.csv"

print("Mapped after LGD village-mapping file:", schools["GP Name"].notna().sum())
print("Still unresolved:", schools["GP Name"].isna().sum())

display(
    schools[schools["Mapping Method"] == "Village->GP via raichur_village_to_gp_mapping.csv"][
        ["UDISE Code", "School Name", "Village / Locality", "GP Name", "Mapping Method"]
    ].head(20)
)

coverage_type values: coverage_type
FULL    861
PART      6
Name: count, dtype: int64
['AMRAPUR', 'DEVERGUD', 'MADARKAL']
Mapped after LGD village-mapping file: 216
Still unresolved: 230


,UDISE Code,School Name,Village / Locality,GP Name,Mapping Method
10,29060104115,Channabasava Lower Primary School Galag,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
15,29060104117,Dr B R Ambedkar Residential School St Galag,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
29,29060104107,Ghs Galag,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
67,29060104101,Gmhigher Primary School Galag,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
70,29060104118,Gnanalahari English Medium Public School,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
93,29060102801,Govt.Higher Primary School Chadkalgudda,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
98,29060103701,Govt.Higher Primary School Devathagal,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
128,29060118101,Govt.Higher Primary School Janmaradi,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
131,29060118201,Govt.Higher Primary School Kajjibandi,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv
213,29060120101,Govt.Lower Primary School Chalwadidoddi,Galag,Galaga,Village->GP via raichur_village_to_gp_mapping.csv


In [ ]:
print("Total schools:", len(schools))
print("Mapped so far:", schools["GP Name"].notna().sum())
print("Still unresolved:", schools["GP Name"].isna().sum())

print("\nBreakdown by Mapping Method:")
print(schools["Mapping Method"].value_counts(dropna=False))

Total schools: 446
Mapped so far: 216
Still unresolved: 230

Breakdown by Mapping Method:
Mapping Method
UNRESOLVED                                           230
Exact village = GP                                   179
Village->GP via raichur_village_to_gp_mapping.csv     19
Exact GP found in address                             18
Name: count, dtype: int64


In [ ]:
unresolved_villages = (
    schools[schools["GP Name"].isna()]["village_norm"]
    .value_counts()
)
print("Distinct unresolved village names:", len(unresolved_villages))
display(unresolved_villages.head(30))

v2g_villages = set(v2g["_village_norm"])

# How many unresolved villages are ALMOST in v2g but not quite matching?
close_misses = [v for v in unresolved_villages.index if v not in v2g_villages]
print("\nUnresolved villages with NO match at all in v2g:", len(close_misses))
print("Unresolved villages that DO exist in v2g (should have matched!):",
      len(unresolved_villages) - len(close_misses))

Distinct unresolved village names: 13


,count
village_norm,
DEVADURGA CITY,33
DEVADURGA SOUTH,26
DEVADURGA WEST,23
SUNKESWARAHAL,21
K IRABAGERA,21
GANADAL,18
KOTTADODDI,16
H SIDDAPUR,16
MALLADEVARAGUDDA,15



Unresolved villages with NO match at all in v2g: 13
Unresolved villages that DO exist in v2g (should have matched!): 0


In [ ]:
# ============================================================
# CELL 6C — SEPARATE "INSIDE DEVADURGA TOWN" FROM GENUINE
#           UNRESOLVED VILLAGES
# ============================================================

TOWN_VILLAGES = ["DEVADURGA CITY", "DEVADURGA SOUTH", "DEVADURGA WEST", "URDU DEVADURGA"]

in_town = schools["GP Name"].isna() & schools["village_norm"].isin(TOWN_VILLAGES)

schools.loc[in_town, "GP Name"] = "NOT APPLICABLE - DEVADURGA TMC"
schools.loc[in_town, "Mapping Method"] = "Inside Devadurga Town Municipal Council (no GP)"

print("Schools tagged as inside Devadurga TMC:", in_town.sum())

# Genuinely unresolved villages remaining
still_unresolved = schools["GP Name"].isna()
remaining_villages = (
    schools[still_unresolved]
    .groupby("village_norm")
    .agg(
        n_schools=("UDISE Code", "count"),
        sample_school=("School Name", "first"),
        sample_address=("Address", "first"),
    )
    .reset_index()
    .sort_values("n_schools", ascending=False)
)

print("\nVillages still needing a manual UDISE+ lookup:", len(remaining_villages))
display(remaining_villages)

# Save this short list as a lookup sheet -- much easier than a 230-row review file
VILLAGE_LOOKUP_FILE = os.path.join(CLEAN_DIR, "Devadurga_Unresolved_Villages_For_Lookup.xlsx")
remaining_villages.to_excel(VILLAGE_LOOKUP_FILE, index=False)
print("\nSaved:", VILLAGE_LOOKUP_FILE)

Schools tagged as inside Devadurga TMC: 92

Villages still needing a manual UDISE+ lookup: 9


,village_norm,n_schools,sample_school,sample_address
4,K IRABAGERA,21,Glps Eligerdoddi,584111) K. Irabagera Devadurga Raichur Karnataka.
8,SUNKESWARAHAL,21,Ghs Hirebudur,584126) Sunkeswarahal Devadurga Raichur Karnat...
2,GANADAL,18,Ghps Palakanamaradi,584115) Ganadal Devadurga Raichur Karnataka.
5,KOTTADODDI,16,Ekalavya Model Residential School Kottadoddi,584111) Kottadoddi Devadurga Raichur Karnataka.
3,H SIDDAPUR,16,Ghs Somanamardi,584116) H Siddapur Devadurga Raichur Karnataka.
6,MALLADEVARAGUDDA,15,Ghps Buddinni,584126) Malladevaragudda Devadurga Raichur Kar...
1,DONDAMBRALLI,15,Govt.Higher Primary School Anchesugur,584111) Dondambralli Devadurga Raichur Karnataka.
7,MASARAKAL,15,Ghs Masarkal,584126) Masarakal Devadurga Raichur Karnataka.
0,,1,Karnataka Public Schools Govt.Lower Primary Sc...,None



Saved: /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/Devadurga_Unresolved_Villages_For_Lookup.xlsx


In [ ]:
# ============================================================
# CELL 6D — APPLY MANUALLY VERIFIED UDISE+ GP LOOKUPS
# ============================================================

manual_gp_lookup = {
    "K IRABAGERA": "K.Irabgera",
    "SUNKESWARAHAL": "Hirebudur",
    "GANADAL": "Palakanamardi",       # FIXED
    "KOTTADODDI": "Kothadoddi",
    "H SIDDAPUR": "Somanamaradi",
    "MALLADEVARAGUDDA": "Jagirajadaladinni",
    "DONDAMBRALLI": "Karigudda",
    "MASARAKAL": "Masarkal",
}

# Match each raw UDISE+ GP name to the closest name
# in your 33 target GPs using the same normalization.
gp_master_lookup_by_norm = dict(
    zip(gp_master["gp_name_norm"], gp_master["gp_name"])
)

resolved_gp_lookup = {}
unmatched = {}

for village, raw_gp in manual_gp_lookup.items():
    norm = normalize_text(raw_gp)

    if norm in gp_master_lookup_by_norm:
        resolved_gp_lookup[village] = gp_master_lookup_by_norm[norm]
    else:
        unmatched[village] = raw_gp

print("Matched directly to one of your 33 target GPs:")

for v, g in resolved_gp_lookup.items():
    print(f"  {v:20s} -> {g}")

if unmatched:
    print("\nWARNING -- these did NOT match any of your 33 target GP names exactly.")
    print("Compare spelling against the list below, then fix manual_gp_lookup and re-run:")

    for v, g in unmatched.items():
        print(f"  {v:20s} -> UDISE+ said '{g}' (no exact match found)")

    display(
        gp_master[["gp_name", "gp_name_norm"]]
        .sort_values("gp_name")
    )

else:
    print("\nAll 8 matched cleanly. Proceed to Cell 6E.")

Matched directly to one of your 33 target GPs:
  K IRABAGERA          -> K.Irabgera
  SUNKESWARAHAL        -> Hirebudur
  GANADAL              -> Palakanamardi
  KOTTADODDI           -> Kothadoddi
  H SIDDAPUR           -> Somanamaradi
  MALLADEVARAGUDDA     -> Jagirajadaladinni
  DONDAMBRALLI         -> Karigudda
  MASARAKAL            -> Masarkal

All 8 matched cleanly. Proceed to Cell 6E.


In [ ]:
# ============================================================
# CELL 6E — APPLY RESOLVED MAPPINGS + TAG FINAL URBAN SCHOOL
# ============================================================

assert not unmatched, "Fix unmatched GP names in Cell 6D before running this."

still_unresolved = schools["GP Name"].isna()

schools.loc[still_unresolved, "GP Name"] = (
    schools.loc[still_unresolved, "village_norm"].map(resolved_gp_lookup)
)

newly_filled = still_unresolved & schools["GP Name"].notna()
schools.loc[newly_filled, "GP LGD Code"] = schools.loc[newly_filled, "GP Name"].map(
    dict(zip(gp_master["gp_name"], gp_master["gp_lgd_code"]))
)
schools.loc[newly_filled, "Mapping Method"] = "Manual UDISE+ LGD Panchayat lookup"

# The one remaining blank-village school (Sipattageri Colony, UDISE 29060122001)
# is inside Devadurga-Municipality -- same treatment as the 92 town schools
still_blank = schools["GP Name"].isna()
schools.loc[still_blank, "GP Name"] = "NOT APPLICABLE - DEVADURGA TMC"
schools.loc[still_blank, "Mapping Method"] = "Inside Devadurga Town Municipal Council (no GP)"

rural_mapped = (schools["GP Name"].notna()) & (schools["GP Name"] != "NOT APPLICABLE - DEVADURGA TMC")
print("Rural schools mapped to a real GP:", rural_mapped.sum())
print("Schools tagged as inside Devadurga TMC (excluded from Need Score):",
      (schools["GP Name"] == "NOT APPLICABLE - DEVADURGA TMC").sum())
print("Still truly unresolved:", schools["GP Name"].isna().sum())

print("\nFull breakdown by Mapping Method:")
print(schools["Mapping Method"].value_counts(dropna=False))

Rural schools mapped to a real GP: 353
Schools tagged as inside Devadurga TMC (excluded from Need Score): 93
Still truly unresolved: 0

Full breakdown by Mapping Method:
Mapping Method
Exact village = GP                                   179
Manual UDISE+ LGD Panchayat lookup                   137
Inside Devadurga Town Municipal Council (no GP)       93
Village->GP via raichur_village_to_gp_mapping.csv     19
Exact GP found in address                             18
Name: count, dtype: int64


## 7. Export the manual-review file

**This is the most important step.** Everything still `UNRESOLVED` goes into an Excel
file, unresolved rows sorted to the top, so you can work through them one by one.

In [ ]:
mapping_review = schools[[
    "UDISE Code", "School Name", "Village / Locality", "Address", "Pincode",
    "Block", "District", "School Category", "State Management",
    "Total Students", "Total Teachers", "GP Name", "GP LGD Code",
    "Mapping Method", "Address GP Candidates"
]].copy()

mapping_review["Review Required"] = mapping_review["GP Name"].isna()

mapping_review = mapping_review.sort_values(
    ["Review Required", "Village / Locality"],
    ascending=[False, True]
)

mapping_review.to_excel(REVIEW_FILE, index=False)

print("Review file created:")
print(REVIEW_FILE)
print("\nMapped:", mapping_review["GP Name"].notna().sum())
print("Need manual verification:", mapping_review["GP Name"].isna().sum())

Review file created:
/content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/Devadurga_School_GP_Mapping_Review.xlsx

Mapped: 446
Need manual verification: 0


## 8. Manual step — resolve `Review Required = TRUE` rows

Do this part **outside the notebook**, in the exported Excel file.

For every row marked `Review Required = TRUE`:

1. Look the school up in **UDISE+ → Know Your School**.
2. Read off the **`LGD Panchayat`** field (not the village name — those often differ).
3. Fill in `GP Name` (and `GP LGD Code` if you have it handy) in the Excel file.

Example, from a real UDISE+ lookup:

```
GOVT HIGHER PRIMARY SCHOOL MADRKAL
LGD Village:    Madarkal
LGD Panchayat:  Shavantagera
```

So `Village = Madarkal` but `GP = Shavantagera` — this is exactly why village name is
**not** a safe stand-in for GP name.

> **Do not do this:**
> ```python
> schools["GP Name"] = schools["Village / Locality"]
> ```
> That silently treats every village as its own Gram Panchayat, which is wrong.

When every row has a `GP Name`, save the file as:

```
Devadurga_School_GP_Mapping_Completed.xlsx
```

in the same `Cleaned/` folder, then continue below.

## 9. Reload the completed mapping file

In [ ]:
import os
for f in sorted(os.listdir(CLEAN_DIR)):
    if "Mapping" in f or "mapping" in f:
        print(f)

Devadurga_School_GP_Mapping_Review.xlsx
raichur_village_to_gp_mapping.csv


In [ ]:
# ============================================================
# CELL 9B — SPLIT RURAL / URBAN, THEN VALIDATE
# ============================================================

mapped_schools = schools[schools["GP Name"] != "NOT APPLICABLE - DEVADURGA TMC"].copy()
urban_excluded = schools[schools["GP Name"] == "NOT APPLICABLE - DEVADURGA TMC"].copy()

print("Rural schools going into GP aggregation:", len(mapped_schools))
print("Urban schools excluded (Devadurga TMC):", len(urban_excluded))
assert len(mapped_schools) + len(urban_excluded) == 446

# --- Validation (Cell 10 from the original notebook) ---

unmapped = mapped_schools[mapped_schools["GP Name"].isna()]
print("\nUnmapped schools:", len(unmapped))

valid_gp_names = set(gp_master["gp_name"])
invalid_gps = mapped_schools[
    ~mapped_schools["GP Name"].isin(valid_gp_names) & mapped_schools["GP Name"].notna()
]
print("Invalid GP assignments:", len(invalid_gps))
if len(invalid_gps) > 0:
    display(invalid_gps[["UDISE Code", "School Name", "GP Name"]])

duplicate_udise = mapped_schools["UDISE Code"].duplicated().sum()
print("Duplicate UDISE records:", duplicate_udise)

gp_code_check = dict(zip(gp_master["gp_name"], gp_master["gp_lgd_code"]))
mapped_schools["Expected GP LGD Code"] = mapped_schools["GP Name"].map(gp_code_check)
wrong_codes = mapped_schools[mapped_schools["GP LGD Code"] != mapped_schools["Expected GP LGD Code"]]
print("Incorrect GP LGD codes:", len(wrong_codes))
if len(wrong_codes) > 0:
    display(wrong_codes[["UDISE Code", "GP Name", "GP LGD Code", "Expected GP LGD Code"]])

Rural schools going into GP aggregation: 353
Urban schools excluded (Devadurga TMC): 93

Unmapped schools: 0
Invalid GP assignments: 0
Duplicate UDISE records: 0
Incorrect GP LGD codes: 0


## 10. Validate the mapping

Run this **before** aggregating. Checks:

- **A.** Every school has a GP
- **B.** Every GP is one of the 33 target GPs
- **C.** UDISE codes are unique
- **D.** `GP LGD Code` matches the code expected for that `GP Name`

You want all four counts to be `0` before moving on.

In [ ]:
# A. Every school should have a GP
unmapped = mapped_schools[mapped_schools["GP Name"].isna()]
print("Unmapped schools:", len(unmapped))
if len(unmapped) > 0:
    print("\nWARNING: Some schools still have no GP.")
    display(unmapped[["UDISE Code", "School Name", "Village / Locality", "Address"]])

# B. Every GP must belong to your 33 target GPs
valid_gp_names = set(gp_master["gp_name"])
invalid_gps = mapped_schools[
    ~mapped_schools["GP Name"].isin(valid_gp_names) & mapped_schools["GP Name"].notna()
]
print("\nInvalid GP assignments:", len(invalid_gps))
if len(invalid_gps) > 0:
    display(invalid_gps[["UDISE Code", "School Name", "GP Name"]])

# C. UDISE code must be unique
duplicate_udise = mapped_schools["UDISE Code"].duplicated().sum()
print("\nDuplicate UDISE records:", duplicate_udise)

# D. GP LGD code consistency
gp_code_check = dict(zip(gp_master["gp_name"], gp_master["gp_lgd_code"]))
mapped_schools["Expected GP LGD Code"] = mapped_schools["GP Name"].map(gp_code_check)
wrong_codes = mapped_schools[mapped_schools["GP LGD Code"] != mapped_schools["Expected GP LGD Code"]]
print("\nIncorrect GP LGD codes:", len(wrong_codes))
if len(wrong_codes) > 0:
    display(wrong_codes[["UDISE Code", "GP Name", "GP LGD Code", "Expected GP LGD Code"]])

# Target result:
# Unmapped schools: 0
# Invalid GP assignments: 0
# Duplicate UDISE records: 0
# Incorrect GP LGD codes: 0

Unmapped schools: 0

Invalid GP assignments: 0

Duplicate UDISE records: 0

Incorrect GP LGD codes: 0


## 11. Aggregate 446 schools -> 33 GP-level education indicators

In [ ]:
df = mapped_schools.copy()

# ---- Management category flags ----
df["is_department_school"] = df["State Management"].eq("1-Department of Education")
df["is_tribal_welfare_school"] = df["State Management"].eq("2-Tribal Welfare Department")
df["is_government_aided_school"] = df["State Management"].eq("4-Government Aided")
df["is_private_school"] = df["State Management"].eq("5-Private Unaided (Recognized)")

# ---- School category flags ----
df["is_primary"] = df["School Category"].eq("1-Primary")
df["is_primary_upper_primary"] = df["School Category"].eq("2-Primary with Upper Primary")
df["is_secondary"] = df["School Category"].eq("8-Secondary Only")
df["is_upper_primary_secondary"] = df["School Category"].eq("7-Upper Pr. and Secondary")
df["is_primary_secondary"] = df["School Category"].eq("6-Pr. Up Pr. and Secondary Only")
df["is_higher_secondary"] = df["School Category"].isin([
    "10-Secondary with Higher Secondary",
    "11-Higher Secondary only/Jr. College",
    "5-Up. Pr. Secondary and Higher Sec",
    "3-Pr. with Up.Pr. Sec. and H.Sec."
])

# ---- Infrastructure flags ----
df["has_library"] = df["Library"].fillna(False).astype(bool)
df["has_drinking_water"] = df["Drinking Water"].fillna(False).astype(bool)
df["has_handwash"] = df["Hand Wash"].fillna(False).astype(bool)
df["has_boundary_wall"] = ~df["Boundary Wall"].astype(str).str.contains("No boundary", case=False, na=False)

# ---- Aggregate to GP level ----
education_gp = (
    df.groupby(["GP Name", "GP LGD Code"], as_index=False)
    .agg(
        total_schools=("UDISE Code", "nunique"),
        total_students=("Total Students", "sum"),
        total_teachers=("Total Teachers", "sum"),
        department_of_education_schools=("is_department_school", "sum"),
        tribal_welfare_schools=("is_tribal_welfare_school", "sum"),
        government_aided_schools=("is_government_aided_school", "sum"),
        private_unaided_schools=("is_private_school", "sum"),
        primary_schools=("is_primary", "sum"),
        primary_upper_primary_schools=("is_primary_upper_primary", "sum"),
        secondary_schools=("is_secondary", "sum"),
        upper_primary_secondary_schools=("is_upper_primary_secondary", "sum"),
        primary_secondary_schools=("is_primary_secondary", "sum"),
        higher_secondary_schools=("is_higher_secondary", "sum"),
        schools_with_library=("has_library", "sum"),
        schools_with_drinking_water=("has_drinking_water", "sum"),
        schools_with_handwash=("has_handwash", "sum"),
        schools_with_boundary_wall=("has_boundary_wall", "sum"),
        boys_toilets=("No. of Boys Toilets", "sum"),
        girls_toilets=("No. of Girls Toilets", "sum"),
        cwsn_toilets=("No. of CWSN Toilets", "sum"),
        classrooms=("Classrooms", "sum"),
    )
)

print("GP-level education table:", education_gp.shape)
display(education_gp.sort_values("GP Name"))

GP-level education table: (20, 23)


,GP Name,GP LGD Code,total_schools,total_students,total_teachers,department_of_education_schools,tribal_welfare_schools,government_aided_schools,private_unaided_schools,primary_schools,...,primary_secondary_schools,higher_secondary_schools,schools_with_library,schools_with_drinking_water,schools_with_handwash,schools_with_boundary_wall,boys_toilets,girls_toilets,cwsn_toilets,classrooms
0,Alkod,219671.0,23,2901,69,18,2,0,3,11,...,0,4,20,23,23,21,37,44,6,129
1,Arkera,219672.0,18,2946,74,13,1,0,4,8,...,0,2,15,18,18,15,32,32,4,152
2,B.Ganekal,219673.0,14,2474,58,13,0,0,1,8,...,0,0,12,14,14,14,20,25,4,91
3,Gabbur,219676.0,26,4855,135,17,2,0,7,7,...,2,2,22,26,26,20,47,42,9,177
4,Galaga,219677.0,19,2295,52,16,1,0,2,12,...,0,1,17,19,19,17,28,30,2,99
5,Gugal,219679.0,14,1762,48,13,0,0,1,5,...,0,0,12,14,14,12,17,18,0,72
6,Hirebudur,219681.0,21,2923,99,16,0,0,5,8,...,0,1,18,21,21,18,28,33,15,148
7,Jagirajadaladinni,219683.0,15,1150,41,14,1,0,0,11,...,0,0,10,15,15,13,17,18,0,57
8,Jalahalli,219684.0,36,6824,153,26,2,2,6,11,...,1,2,29,36,36,33,57,52,30,243
9,K.Irabgera,219686.0,21,1895,56,18,0,0,3,16,...,1,0,20,21,21,20,25,25,4,61


## 12. Derived indicators

Raw counts alone are less useful for the Need Score than rates/coverage. Note:
`Drinking Water` and `Hand Wash` were TRUE for all 446 schools in the source file, so
they have zero variation — not useful Need Score indicators. Library, boundary-wall
status, student/teacher ratio and school-level mix have more variation and are more
informative.

In [ ]:
education_gp["students_per_teacher"] = (
    education_gp["total_students"] / education_gp["total_teachers"].replace(0, np.nan)
)

education_gp["schools_per_1000_students"] = (
    education_gp["total_schools"] / education_gp["total_students"].replace(0, np.nan)
) * 1000

education_gp["library_coverage_pct"] = (
    education_gp["schools_with_library"] / education_gp["total_schools"]
) * 100

education_gp["boundary_wall_coverage_pct"] = (
    education_gp["schools_with_boundary_wall"] / education_gp["total_schools"]
) * 100

education_gp["drinking_water_coverage_pct"] = (
    education_gp["schools_with_drinking_water"] / education_gp["total_schools"]
) * 100

education_gp["handwash_coverage_pct"] = (
    education_gp["schools_with_handwash"] / education_gp["total_schools"]
) * 100

display(education_gp.head())

,GP Name,GP LGD Code,total_schools,total_students,total_teachers,department_of_education_schools,tribal_welfare_schools,government_aided_schools,private_unaided_schools,primary_schools,...,boys_toilets,girls_toilets,cwsn_toilets,classrooms,students_per_teacher,schools_per_1000_students,library_coverage_pct,boundary_wall_coverage_pct,drinking_water_coverage_pct,handwash_coverage_pct
0,Alkod,219671.0,23,2901,69,18,2,0,3,11,...,37,44,6,129,42.043478,7.928301,86.956522,91.304348,100.0,100.0
1,Arkera,219672.0,18,2946,74,13,1,0,4,8,...,32,32,4,152,39.810811,6.109980,83.333333,83.333333,100.0,100.0
2,B.Ganekal,219673.0,14,2474,58,13,0,0,1,8,...,20,25,4,91,42.655172,5.658852,85.714286,100.000000,100.0,100.0
3,Gabbur,219676.0,26,4855,135,17,2,0,7,7,...,47,42,9,177,35.962963,5.355304,84.615385,76.923077,100.0,100.0
4,Galaga,219677.0,19,2295,52,16,1,0,2,12,...,28,30,2,99,44.134615,8.278867,89.473684,89.473684,100.0,100.0


## 13. Confirm all 33 target GPs are covered

If this comes up short of 33, **do not fill the gaps with zero** — it means those GPs
still need school→GP mapping work (go back to step 8).

In [ ]:
education_gp_names = set(education_gp["GP Name"])
target_gp_names = set(gp_master["gp_name"])

missing_gps = sorted(target_gp_names - education_gp_names)
extra_gps = sorted(education_gp_names - target_gp_names)

print("Target GPs:", len(target_gp_names))
print("Education GPs:", len(education_gp_names))
print("\nMissing GPs:", missing_gps)
print("Unexpected GPs:", extra_gps)

# Desired result:
# Target GPs: 33
# Education GPs: 33
# Missing GPs: []
# Unexpected GPs: []

Target GPs: 33
Education GPs: 20

Missing GPs: ['Amarapur', 'Bhumanagunda', 'Chinchodi', 'Dondambli', 'Ganadhal (Deodurga)', 'Hemanal', 'Hosur Siddapur', 'Jeerabandi', 'Karadigudda', 'Maladakal', 'Malledevaragudda', 'Ramadurga', 'Shavantagera']
Unexpected GPs: []


## 14. Save the final GP-level education table

In [ ]:
education_gp.to_csv(EDUCATION_OUTPUT, index=False)

print("Saved:")
print(EDUCATION_OUTPUT)

Saved:
/content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/devadurga_gp_education_data.csv


---

## Next step (back in `02_gp_level_join.ipynb`)

Load `devadurga_gp_education_data.csv` and merge it into the master table on
`gp_lgd_code` (not on GP name):

```python
education_gp = pd.read_csv(EDUCATION_OUTPUT)

education_columns = [c for c in education_gp.columns if c not in ["GP Name", "GP LGD Code"]]
merged = merged.drop(columns=[c for c in education_columns if c in merged.columns], errors="ignore")

merged = merged.merge(
    education_gp,
    left_on="gp_lgd_code",
    right_on="GP LGD Code",
    how="left",
    validate="one_to_one"
)

if "GP Name" in merged.columns:
    merged = merged.drop(columns=["GP Name"])

print("Final master shape:", merged.shape)
```

### Full pipeline

```
                446 SCHOOL RECORDS
                        |
              School -> GP Mapping
                        |
                33 GRAM PANCHAYATS
                        |
              Education Aggregation
                        |
   Total Schools / Government / Private / Primary / Secondary /
   Higher Secondary / Students / Teachers / Students per Teacher /
   Library Coverage / Boundary Wall Coverage / Drinking Water /
   Toilets / Classrooms
                        |
                  MASTER GP TABLE
                        |
               NEED SCORE / ENTROPY
```

In [ ]:
# ============================================================
# DIAGNOSTIC — INVESTIGATE THE 13 MISSING GPs
# ============================================================

education_gp = pd.read_csv(EDUCATION_OUTPUT)

target_gp_names = set(gp_master["gp_name"])
achieved_gp_names = set(education_gp["GP Name"])

missing_gps = sorted(target_gp_names - achieved_gp_names)

print("Missing GPs:", len(missing_gps))
print(missing_gps)

for gp in missing_gps:
    gp_norm = normalize_text(gp)

    hits = schools[
        schools["address_norm"].str.contains(gp_norm, na=False)
    ]

    print(
        f"\n{gp!r} ({gp_norm}) -- "
        f"mentioned in {len(hits)} school addresses:"
    )

    if len(hits) > 0:
        display(
            hits[
                [
                    "UDISE Code",
                    "School Name",
                    "Village / Locality",
                    "Address",
                    "GP Name",
                    "Mapping Method",
                ]
            ]
        )

Missing GPs: 13
['Amarapur', 'Bhumanagunda', 'Chinchodi', 'Dondambli', 'Ganadhal (Deodurga)', 'Hemanal', 'Hosur Siddapur', 'Jeerabandi', 'Karadigudda', 'Maladakal', 'Malledevaragudda', 'Ramadurga', 'Shavantagera']

'Amarapur' (AMARAPUR) -- mentioned in 0 school addresses:

'Bhumanagunda' (BHUMANAGUNDA) -- mentioned in 0 school addresses:

'Chinchodi' (CHINCHODI) -- mentioned in 0 school addresses:

'Dondambli' (DONDAMBLI) -- mentioned in 0 school addresses:

'Ganadhal (Deodurga)' (GANADHAL DEODURGA) -- mentioned in 0 school addresses:

'Hemanal' (HEMANAL) -- mentioned in 0 school addresses:

'Hosur Siddapur' (HOSUR SIDDAPUR) -- mentioned in 0 school addresses:

'Jeerabandi' (JEERABANDI) -- mentioned in 0 school addresses:

'Karadigudda' (KARADIGUDDA) -- mentioned in 0 school addresses:

'Maladakal' (MALADAKAL) -- mentioned in 0 school addresses:

'Malledevaragudda' (MALLEDEVARAGUDDA) -- mentioned in 0 school addresses:

'Ramadurga' (RAMADURGA) -- mentioned in 0 school addresses:

'Shav

In [ ]:
# ============================================================
# DIAGNOSTIC 2 — CHECK VILLAGES BELONGING TO THE 13 MISSING GPs
# ============================================================

missing_gp_set = set(missing_gps)

# Load village → GP mapping
village_gp = pd.read_csv(
    os.path.join(CLEAN_DIR, "raichur_village_to_gp_mapping.csv")
)

print("Village-GP mapping columns:")
print(village_gp.columns.tolist())

display(village_gp.head())

Village-GP mapping columns:
['sub_taluk', 'village_code', 'village_name', 'coverage_type', 'gp_lgd_code', 'gp_name']


,sub_taluk,village_code,village_name,coverage_type,gp_lgd_code,gp_name
0,Arakera,600702,Adkalgud,FULL,273066,Bhumanagunda
1,Arakera,600715,Agrahar,FULL,273067,Malledevaragudda
2,Arakera,600704,Akalkumpi,FULL,219671,Alkod
3,Arakera,600660,Aldharti,FULL,219695,Mustur
4,Arakera,600706,Alkod,FULL,219671,Alkod


In [ ]:
# ============================================================
# DIAGNOSTIC 3 — CHECK VILLAGE COVERAGE FOR THE 13 MISSING GPs
# ============================================================

# Normalize village names from the official village → GP mapping
village_gp["village_name_norm"] = (
    village_gp["village_name"]
    .astype(str)
    .map(normalize_text)
)

# Normalize school village/locality names
schools["village_norm"] = (
    schools["Village / Locality"]
    .fillna("")
    .astype(str)
    .map(normalize_text)
)

# Get all official villages belonging to the 13 missing GPs
missing_villages = village_gp[
    village_gp["gp_name"].isin(missing_gps)
].copy()

print("Official villages belonging to the 13 missing GPs:")
print("Total villages:", len(missing_villages))

display(
    missing_villages[
        ["gp_name", "village_name", "village_code", "gp_lgd_code"]
    ].sort_values(["gp_name", "village_name"])
)

Official villages belonging to the 13 missing GPs:
Total villages: 85


,gp_name,village_name,village_code,gp_lgd_code
91,Amarapur,Amrapur,600559,273065
114,Amarapur,Gajaldinni,600562,273065
138,Amarapur,Jambaldinni,600546,273065
155,Amarapur,Mandalgud,600561,273065
163,Amarapur,Mukanhal,600563,273065
...,...,...,...,...
123,Shavantagera,Hanchinhal,600650,273068
131,Shavantagera,Hireraikumpi,600639,273068
135,Shavantagera,Ingaladhal,600656,273068
154,Shavantagera,Madarkal,600640,273068


In [ ]:
# ============================================================
# DIAGNOSTIC 4 — MATCH OFFICIAL VILLAGES AGAINST SCHOOL DATA
# ============================================================

# Exact village-name matches
village_matches = missing_villages[
    missing_villages["village_name_norm"].isin(
        set(schools["village_norm"])
    )
].copy()

print("============================================================")
print("VILLAGES FROM MISSING GPs FOUND IN SCHOOL DATA")
print("============================================================")

print("Matched villages:", len(village_matches))

display(
    village_matches[
        ["gp_name", "village_name", "village_code", "gp_lgd_code"]
    ].sort_values(["gp_name", "village_name"])
)

VILLAGES FROM MISSING GPs FOUND IN SCHOOL DATA
Matched villages: 0


,gp_name,village_name,village_code,gp_lgd_code


In [ ]:
# ============================================================
# DIAGNOSTIC 5 — SHOW THE ACTUAL SCHOOLS IN THOSE VILLAGES
# ============================================================

matched_village_norms = set(
    village_matches["village_name_norm"]
)

school_hits = schools[
    schools["village_norm"].isin(matched_village_norms)
].copy()

print("Schools found in villages belonging to missing GPs:", len(school_hits))

if len(school_hits) > 0:
    display(
        school_hits[
            [
                "UDISE Code",
                "School Name",
                "Village / Locality",
                "Address",
                "GP Name",
                "Mapping Method"
            ]
        ].sort_values(["Village / Locality", "School Name"])
    )
else:
    print("No schools found in any official village belonging to the 13 missing GPs.")

Schools found in villages belonging to missing GPs: 0
No schools found in any official village belonging to the 13 missing GPs.


In [ ]:
# ============================================================
# CELL 13B — DOCUMENT THE 13 GPs WITH NO SCHOOL-LEVEL DATA
# ============================================================

no_school_data_gps = pd.DataFrame({"gp_name": missing_gps})
no_school_data_gps = no_school_data_gps.merge(gp_master[["gp_name", "gp_lgd_code"]], on="gp_name", how="left")
no_school_data_gps["note"] = "No schools found in 446-school source file (verified: 0 exact village matches, 0 address mentions)"

NO_DATA_FILE = os.path.join(CLEAN_DIR, "Devadurga_GPs_No_Education_Data.csv")
no_school_data_gps.to_csv(NO_DATA_FILE, index=False)

print("Documented", len(no_school_data_gps), "GPs with no school-level data.")
print("Saved:", NO_DATA_FILE)
display(no_school_data_gps)

Documented 13 GPs with no school-level data.
Saved: /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/Devadurga_GPs_No_Education_Data.csv


,gp_name,gp_lgd_code,note
0,Amarapur,273065,No schools found in 446-school source file (ve...
1,Bhumanagunda,273066,No schools found in 446-school source file (ve...
2,Chinchodi,219674,No schools found in 446-school source file (ve...
3,Dondambli,219675,No schools found in 446-school source file (ve...
4,Ganadhal (Deodurga),219678,No schools found in 446-school source file (ve...
5,Hemanal,219680,No schools found in 446-school source file (ve...
6,Hosur Siddapur,219682,No schools found in 446-school source file (ve...
7,Jeerabandi,219685,No schools found in 446-school source file (ve...
8,Karadigudda,219687,No schools found in 446-school source file (ve...
9,Maladakal,219692,No schools found in 446-school source file (ve...
